In [3]:
%pip install -q "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 27.4 MB/s eta 0:00:00


In [4]:
from mcp.server import MCPServer

server = MCPServer(
    "MathServer",
    instructions="Provides safe basic mathematical operations."
)

@server.tool()
def add_numbers(a:int,b:int)-> int:
  """Add two integers and return the result."""
  return a + b

@server.tool()
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

@server.resource("math://constants")
def math_constants() -> str:
    """Return a few mathematical constants."""
    return "pi ≈ 3.14159; e ≈ 2.71828"

@server.prompt()
def explain_calculation(expression: str) -> str:
    """Create a prompt to explain a mathematical expression."""
    return (
        "Explain the following calculation step by step: "
        + expression
    )

In [5]:
#Connect an in-process client

import asyncio
from mcp import Client

async def test_mcp_client():
    async with Client(server) as client:
        print("Server:", client.server_info)
        print("Capabilities:", client.server_capabilities)

        tools = await client.list_tools()
        print("\nAvailable tools:")
        for tool in tools.tools:
            print("-", tool.name, ":", tool.description)

        result = await client.call_tool(
            "add_numbers",
            {"a": 500, "b": 300}
        )

        print("\nTool result:")
        print(result)

await test_mcp_client()

Server: name='MathServer' title=None version='' description=None website_url=None icons=None
Capabilities: experimental=None logging=None prompts=PromptsCapability(list_changed=True) resources=ResourcesCapability(subscribe=True, list_changed=True) tools=ToolsCapability(list_changed=True) completions=None extensions=None tasks=None

Available tools:
- add_numbers : Add two integers and return the result.
- multiply_numbers : Multiply two integers and return the result.

Tool result:
meta={'io.modelcontextprotocol/serverInfo': {'name': 'MathServer', 'version': ''}} content=[TextContent(type='text', text='800', annotations=None, meta=None)] structured_content={'result': 800} is_error=False result_type='complete'


In [6]:
#Call the multiplication tool

async def test_server_tool():
  async with Client(server) as client:
    result = await client.call_tool(
        "multiply_numbers",
            {"a": 12, "b": 8}
    )
    print(result)

await test_server_tool()

meta={'io.modelcontextprotocol/serverInfo': {'name': 'MathServer', 'version': ''}} content=[TextContent(type='text', text='96', annotations=None, meta=None)] structured_content={'result': 96} is_error=False result_type='complete'


In [7]:
#Tool implementation

INCIDENTS = [
    {
        "id": "INC-101",
        "service": "payment-api",
        "severity": "high",
        "description": "Payment requests are timing out."
    },
    {
        "id": "INC-102",
        "service": "auth-service",
        "severity": "medium",
        "description": "Login latency has increased."
    },
    {
        "id": "INC-103",
        "service": "payment-api",
        "severity": "critical",
        "description": "Payment failures increased after deployment."
    }
]

@server.tool()

def search_incidents(service_name: str) -> list[dict]:
  """Search incident records by service name."""
  query = service_name.strip().lower()

  if not query:
    raise ValueError("service_name must not be empty")

  return [
      incident
      for incident in INCIDENTS
      if query in incident["service"].lower()
  ]


In [11]:
async def test_incident_search():
    async with Client(server) as client:
        result = await client.call_tool(
            "search_incidents",
            {"service_name": "payment-api"}
        )

        for content in result.content:
            if hasattr(content, "text"):
                print(content.text)

        if result.isError:
            print("The tool reported an error.")

await test_incident_search()

{
  "id": "INC-101",
  "service": "payment-api",
  "severity": "high",
  "description": "Payment requests are timing out."
}
{
  "id": "INC-103",
  "service": "payment-api",
  "severity": "critical",
  "description": "Payment failures increased after deployment."
}


ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

In [12]:
#security exercise , Add an allowlist

ALLOWED_SERVICES = {
    "payment-api",
    "auth-service"
}

@server.tool()
def search_incidents_secure(service_name: str) -> list[dict]:
    """Search incidents for an explicitly allowed service."""

    service = service_name.strip().lower()

    if service not in ALLOWED_SERVICES:
        raise ValueError("Service is not allowed.")

    return [
        incident
        for incident in INCIDENTS
        if incident["service"] == service
    ]

In [ ]:
#Test permitted and rejected inputs

async def test_security():
    async with Client(server) as client:
        allowed = await client.call_tool(
            "search_incidents_secure",
            {"service_name": "payment-api"}
        )

        print("Allowed call:", allowed)

        rejected = await client.call_tool(
            "search_incidents_secure",
            {"service_name": "unknown-service"}
        )

        print("Rejected call error:", rejected.isError)

await test_security()

In [14]:
#Connecting an MCP tool to an external REST API

%pip install -q requests

In [15]:
import requests

@server.tool()
def get_post(post_id: int) -> dict:
    """Fetch a demonstration post from a public REST API."""

    if not 1 <= post_id <= 100:
        raise ValueError("post_id must be between 1 and 100")

    url = (
        "https://jsonplaceholder.typicode.com/posts/"
        f"{post_id}"
    )

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()

    return {
        "id": data["id"],
        "title": data["title"],
        "body": data["body"]
    }

In [ ]:
async def test_external_api():
    async with Client(server) as client:
        result = await client.call_tool(
            "get_post",
            {"post_id": 1}
        )

        for item in result.content:
            if hasattr(item, "text"):
                print(item.text)

        if result.isError:
            print("External API tool failed.")

await test_external_api()